# 👥 04 - Comparing Groups

Module 06 taught the mechanics of `groupby()`, `.agg()`, and `value_counts()`. This notebook uses them for the most common question in data analysis: **"Is this group different from that one?"**

Comparing groups well takes more than the right method call. You need the right **statistic** for the column type (notebook 01), a fair **denominator** (counts vs. rates), enough **data** in each group, and an eye out for a **second variable** hiding behind the first.

## ✅ Learning Goals
- Summarize a numeric column by group with center **and** spread, plus the group size
- Compare a yes/no outcome across groups as a **rate**, not a count
- Build two-way tables with `pd.crosstab()` (counts and row percentages) and `pivot_table()`
- Compare groups made from a numeric column by binning it with `pd.cut()`
- Spot small groups and confounding variables before drawing conclusions

In [ ]:
import pandas as pd

titanic = pd.read_csv('titanic.csv')
titanic['class'] = titanic['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})

## 1. Numeric Column by Group: Center, Spread, and Size
A group mean on its own hides too much. A useful group summary always has **how many**, **where the middle is**, and **how spread out**. Named aggregation (Module 06) keeps the column names readable:

In [ ]:
fare_by_class = (
    titanic
    .groupby('class')
    .agg(
        passengers=('Fare', 'size'),
        mean_fare=('Fare', 'mean'),
        median_fare=('Fare', 'median'),
        std_fare=('Fare', 'std'),
        max_fare=('Fare', 'max'),
    )
    .round(2)
)
fare_by_class

Read it with notebooks 02 and 03 in mind. 1st class has the biggest gap between mean and median, and by far the biggest std: the outlier tickets from notebook 03 all live in this group. To compare **typical** fares across classes, the median is the fairer choice.

### Quick Version: `.describe()` by Group
`groupby(...)[col].describe()` gives the full `.describe()` table for each group in one line:

In [ ]:
titanic.groupby('class')['Age'].describe().round(1)

✅ **Your Turn:** Build a named-aggregation table of `Age` by `Sex` with the count of **known** ages (`'count'`), the mean, the median, and the std. Why does `'count'` differ from `'size'` here? (Module 06's `.size()` vs. `.count()`.)

In [ ]:
# Your turn here

## 2. Yes/No Outcome by Group: Rates, Not Counts
Notebook 01 showed that the mean of a 0/1 column is a proportion. Group by something, take the mean of `Survived`, and you get a **survival rate per group**:

In [ ]:
titanic.groupby('Sex')['Survived'].agg(['size', 'sum', 'mean']).rename(
    columns={'size': 'passengers', 'sum': 'survived', 'mean': 'survival_rate'}
)

Look at the `survived` and `survival_rate` columns together. 233 women and 109 men survived. Comparing **counts** would already say "women did better," but only because the counts happen to agree with the rates. Counts depend on how big each group is. Rates divide that out.

Here's a case where counts mislead:

In [ ]:
by_class = titanic.groupby('class')['Survived'].agg(['size', 'sum', 'mean']).rename(
    columns={'size': 'passengers', 'sum': 'survived', 'mean': 'survival_rate'}
)
by_class

More 3rd-class passengers survived (119) than 2nd-class (87). But 3rd class was much bigger, and its survival **rate** was the lowest. "More survivors" and "better chances" are different claims.

> 🧠 Before comparing groups, ask: **is the denominator the same?** If not, compare rates or percentages.

## 3. Two Categorical Columns: `pd.crosstab()`
`pd.crosstab(rows, columns)` counts every combination of two categorical columns:

In [ ]:
pd.crosstab(titanic['class'], titanic['Sex'], margins=True)

`margins=True` adds an `All` row and column of totals. Counts again have the denominator problem, so use `normalize=` to turn them into percentages:

| `normalize=` | Each … adds up to 100% | Answers |
|---|---|---|
| `'index'` | row | "Within each class, what share is female?" |
| `'columns'` | column | "Of all the women, what share was in each class?" |
| `'all'` | whole table | "What share of everyone is a 1st-class woman?" |

In [ ]:
(pd.crosstab(titanic['class'], titanic['Sex'], normalize='index') * 100).round(1)

In [ ]:
(pd.crosstab(titanic['class'], titanic['Sex'], normalize='columns') * 100).round(1)

Same data, two different questions, two different tables. Before running a crosstab, say the question out loud. The question decides which way to normalize.

## 4. A Statistic for Every Combination: `pivot_table()`
`crosstab` counts. `pivot_table` computes **any** statistic of a third column, for every combination of two others:

In [ ]:
survival_table = titanic.pivot_table(
    values='Survived',
    index='class',
    columns='Sex',
    aggfunc='mean',
)
(survival_table * 100).round(1)

This table says much more than either one-way table. Women in 1st and 2nd class almost all survived. Men in 2nd and 3rd class almost all didn't. The class gap is large for women (97% vs. 50%) and the sex gap is large in every class.

In Module 08, this exact table becomes a **heatmap**.

## 5. Groups from a Numeric Column: `pd.cut()`
To compare age groups, first turn `Age` into bands with `pd.cut()` (Module 06), then group by the bands.

In [ ]:
titanic['age_group'] = pd.cut(
    titanic['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['child', 'teen', 'young adult', 'adult', 'senior'],
)

titanic.groupby('age_group', observed=True)['Survived'].agg(['size', 'mean']).round(3)

`pd.cut()` returns an **ordered category** (notebook 01), so the groups come back in age order. `observed=True` tells pandas to show only the bands that actually have rows.

⚠️ The sizes add up to 714, not 891. Passengers with no recorded age aren't in any band. A survival rate for "children" says nothing about the 177 passengers whose age is unknown.

## 6. Comparing Fairly: Two Traps
### Trap 1: Tiny Groups
A rate from 5 people is much less trustworthy than a rate from 500. Always keep the group size next to the statistic, and be suspicious of extreme rates from small groups:

In [ ]:
family = titanic.groupby('SibSp')['Survived'].agg(['size', 'mean']).round(2)
family

The `5` and `8` rows show a 0% survival rate, but they're tiny groups (each one is essentially a single family). One family's fate isn't a pattern. A common fix is to only report groups above a minimum size, using Module 06's group filtering:

In [ ]:
family[family['size'] >= 20]

### Trap 2: A Hidden Second Variable (Confounding)
Passengers who boarded at Cherbourg (`C`) had the highest survival rate:

In [ ]:
titanic.groupby('Embarked')['Survived'].agg(['size', 'mean']).round(3)

Did boarding at Cherbourg save lives? Check who boarded there:

In [ ]:
(pd.crosstab(titanic['Embarked'], titanic['class'], normalize='index') * 100).round(1)

About half of the Cherbourg passengers were in 1st class, far more than at the other ports. Class affected survival, and class also differed by port. So the port's survival rate is partly a **class** effect in disguise. A variable like `class` that's tangled up with both the group and the outcome is a **confounder**.

The fix is to compare *within* levels of the suspected confounder:

In [ ]:
(titanic.pivot_table(values='Survived', index='class', columns='Embarked', aggfunc='mean') * 100).round(1)

Within each class the port differences are smaller, though not all gone. Grouping alone can't prove what *caused* a difference. It can only show you where to look. Notebook 05 says the same thing about correlation.

---
### 📝 Summary
- Report **size, center, and spread** for every group, not just the mean.
- Compare yes/no outcomes as **rates** (`groupby(...)['col'].mean()`), not counts.
- `pd.crosstab()` counts combinations. `normalize='index'` / `'columns'` answers different questions.
- `pivot_table()` computes any statistic for every combination of two columns.
- Watch for **small groups** (unreliable rates) and **confounders** (a second variable behind the difference).

## 🏋️ Practice

### Practice 1 — Legendary vs. Not (easy)
Load `pokemon.csv` into `poke`. Compare `Total` for Legendary and non-Legendary Pokémon with a named-aggregation table showing the count, mean, median, and std.

In [ ]:
poke = pd.read_csv('pokemon.csv')
# Your code here

### Practice 2 — Legendary Rate by Generation (medium)
Find the share of Pokémon that are Legendary in each `Generation`, as a percentage. Then make a `pd.crosstab()` of `Generation` vs. `Legendary` with `margins=True`. Which generation has the highest Legendary **rate**, and is it also the one with the most Legendary Pokémon?

In [ ]:
# Your code here

### Practice 3 — Fair Type Rankings (harder)
Rank `Type 1` by median `Attack`, but **only for types with at least 30 Pokémon**. Show the size and median for the top 5. Then show which types the minimum-size rule removed, and why including them could have been misleading.

In [ ]:
# Your code here

*Your explanation here.*

## ✏️ Your Turn
Make a `pivot_table()` of the **median** `Fare` by `class` (rows) and `Embarked` (columns). In a sentence or two, explain how this table supports the confounding story from section 6.

In [ ]:
# Your turn here

*Your explanation here.*

## 🔥 Challenge (Optional) — Differences Between Groups
"Women survived more" is vague. Make it precise. From `survival_table` (section 4), compute for each class:
1. the **difference** in survival rates (women − men), in percentage points, and
2. the **ratio** of the rates (women ÷ men).

In which class is the gap biggest by each measure? Why can the two measures disagree?

In [ ]:
# Your code here

---
✅ Next up → [Relationships and Correlation](05-Relationships_and_Correlation.ipynb): comparing two **numeric** columns, and the limits of a single correlation number.